# GLOBAL RICE TRADE NETWORK
A network-based analysis of the global rice trade to identify structural hubs, chokepoints, and systemic fragility — with a focus on Vietnam's position.

---

## Research Question
*Which countries form the hubs of the global rice trade network, where are the
chokepoints, how fragile is the system — and where does Vietnam sit within it?*

## Approach

The global rice trade is modeled as a **directed, weighted network** (nodes =
countries, edges = bilateral export flows). Structural metrics — centrality,
concentration, and targeted node removal — are used to assess how dependent and
fragile the system is.

## Data Sources

| Source | Description |
|--------|-------------|
| FAOSTAT Detailed Trade Matrix | Bilateral rice export flows, 2010–2024 |
| World Bank Pink Sheet | Monthly international rice prices |

## 1. Load raw data

Read the two raw files into pandas: the trade matrix (network edges) and the
monthly price series.

In [33]:
import pandas as pd
import sqlite3

# Load the trade file (rice trade network data)
trade = pd.read_csv("data/raw/rice_trade_2010_2024.csv")

print("Shape (rows, cols): ", trade.shape)
print("\nColumn names:")
print(trade.columns.tolist())
print("\nFirst 5 rows:")
trade.head()

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")





Shape (rows, cols):  (7424, 20)

Column names:
['Reporter Countries', 'Partner Countries', 'Item', 'Element', 'Unit', 'Y2010', 'Y2011', 'Y2012', 'Y2013', 'Y2014', 'Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020', 'Y2021', 'Y2022', 'Y2023', 'Y2024']

First 5 rows:


## 2. Store data in a SQLite database

Save both tables into a local SQLite database (`rice.db`) so we can query them
with SQL and reuse them across notebooks.

In [34]:
# Load the World Bank Pink Sheet (monthly rice prices)
# Commodity names are on row 5, so we skip the first 4 rows. 
prices = pd.read_excel("data/raw/CMO-Historical-Data-Monthly.xlsx", sheet_name = "Monthly Prices", skiprows=4)
# Rename the first column (the month column)
prices = prices.rename (columns = {prices.columns[0]: "Month"})
print("Shape (rows, cols):", prices.shape)
print("\nRice-related columns:")
print([c for c in prices.columns if "Rice" in str(c)])
print("\nLast 5 rows (Month + Thai 5% rice):")
prices[["Month", "Rice, Thai 5% "]].tail()


Shape (rows, cols): (799, 72)

Rice-related columns:
['Rice, Thai 5% ', 'Rice, Thai 25% ', 'Rice, Thai A.1', 'Rice, Viet Namese 5%']

Last 5 rows (Month + Thai 5% rice):


,Month,"Rice, Thai 5%"
794,2026M02,409
795,2026M03,381
796,2026M04,403
797,2026M05,440
798,2026M06,494


In [35]:
# Create/Connect to a SQLite database file inside data/
my_db = sqlite3.connect("data/rice.db")
# Write both tables into the database
trade.to_sql("trade", my_db, if_exists= "replace", index = False)
prices.to_sql("prices",my_db, if_exists= "replace", index = False)
my_db.commit()
print ("Saved 2 tables into data/rice.db")
# Check: list the tables now inside the database
tables = pd.read_sql("SELECT name from sqlite_master WHERE type = 'table';", my_db)
print("\nTables in the database:")
print(tables)



Saved 2 tables into data/rice.db

Tables in the database:
          name
0  trade_clean
1        trade
2       prices


## 3. Clean the trade data

The trade matrix is in **wide format** (one column per year), which is awkward
to analyze. We reshape it to **long format** (one row per reporter–partner–year),
convert the year to an integer, and drop rows with no trade value (NaN =
no trade that year).

In [36]:
# Build the list of year columns
year_cols = [f"Y{y}" for y in range (2010, 2025)]
# Reshape trade data from wide to long format
trade_long = trade.melt(id_vars=["Reporter Countries", "Partner Countries"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)
print ("New shape:", trade_long.shape)
trade_long.head()




New shape: (111360, 4)


,Reporter Countries,Partner Countries,Year,Value
0,Afghanistan,Germany,Y2010,NaN
1,Afghanistan,India,Y2010,NaN
2,Afghanistan,Pakistan,Y2010,NaN
3,Afghanistan,Sweden,Y2010,NaN
4,Afghanistan,Tajikistan,Y2010,NaN


In [37]:
# Convert Year from text to integer
trade_long["Year"] = trade_long["Year"].str.replace("Y", "").astype(int)
# Drop rows with no trade value (NaN)
trade_long = trade_long.dropna(subset=["Value"])
print("Shape after cleaning:", trade_long.shape)
trade_long.head()


Shape after cleaning: (37886, 4)


,Reporter Countries,Partner Countries,Year,Value
30,Argentina,Angola,2010,"1,722"
34,Argentina,Belgium,2010,100
37,Argentina,Bolivia (Plurinational State of),2010,"1,319"
38,Argentina,Brazil,2010,"171,961"
41,Argentina,Canada,2010,40


In [38]:
# Fix character-encoding artifacts in country names
name_fixes = {
    "CÃ´te d'Ivoire": "Côte d'Ivoire",
    "TÃ¼rkiye": "Türkiye",
}
trade_long["Reporter Countries"] = trade_long["Reporter Countries"].replace(name_fixes)
trade_long["Partner Countries"]  = trade_long["Partner Countries"].replace(name_fixes)

# Check leftover odd characters
mask = trade_long["Partner Countries"].str.contains("Ã", na=False)
print("Rows still with encoding issues:", mask.sum())
print(trade_long.loc[mask, "Partner Countries"].unique())

Rows still with encoding issues: 0
<StringArray>
[]
Length: 0, dtype: str


In [39]:
# Save the cleaned long-format trade data back into the database
trade_long.to_sql("trade_clean", my_db, if_exists="replace", index = False)
my_db.commit()
print ("Saved cleaned table 'trade_clean':", trade_long.shape)

Saved cleaned table 'trade_clean': (37886, 4)


## 4. Exploratory Analysis: Top Rice Importers
Which countries import the most rice over 2010 - 2024?
This gives a first look at demand-side concentration in the network. 

In [40]:
query = """
SELECT "Partner Countries", SUM("Value") AS total_import
FROM trade_clean
GROUP BY "Partner Countries"
ORDER BY total_import DESC
LIMIT 10;
"""
top_partners = pd.read_sql(query, my_db)
print(top_partners)

            Partner Countries  total_import
0             China, mainland    28,831,078
1                       Benin    25,815,066
2                 Philippines    25,653,147
3                Saudi Arabia    20,376,611
4                        Iraq    19,334,916
5  Iran (Islamic Republic of)    17,466,399
6               Côte d'Ivoire    16,935,744
7                    Malaysia    14,818,179
8                South Africa    13,936,755
9                   Indonesia    13,754,293


**Observations - Top Rice Importers:**
- Import demand is relatively dispersed — The largest importer (China, 28.8M) is only about twice the tenth (Indonesia, 13.7M).
- Importers are **geographically diverse**: East and Southeast Asia (China, Philippines, Malaysia, Indonesia), Middle East (Saudi Arabia, Iraq, Iran), West Africa (Benin, Côte d'Ivoire), and Southern Africa (South Africa).
- Benin stands out as a possible re-export hub: A small country importing large volumes, likely re-routing rice onward. This is worth revisiting in the chokepoint analysis.

**Caveat:** Totals are summed over 2010–2024, so they reflect cumulative volume,
not current or year-by-year patterns.
    

In [41]:
query = """
SELECT "Reporter Countries", SUM("Value") AS total_export
FROM trade_clean
GROUP BY "Reporter Countries"
ORDER BY total_export DESC
LIMIT 10;
"""
top_exporters = pd.read_sql(query, my_db)
print(top_exporters)

         Reporter Countries  total_export
0                     India   165,752,874
1                  Thailand   114,504,826
2                  Viet Nam    65,979,907
3                  Pakistan    47,753,021
4  United States of America    26,154,389
5           China, mainland    15,804,390
6                     Italy     8,917,588
7                   Uruguay     8,487,750
8                  Cambodia     7,023,227
9                    Brazil     5,246,727


**Observations - Top Rice Exporters:**
- Supply is **highly concentrated** — India, Thailand and Vietnam dominate, and the top exporter (India, 165M) is about 33 times the tenth (Brazil, 5M).
- This contrasts sharply with the dispersed *import* side.
- **Concentrated supply + dispersed demand = systemic risk**. A disruption at one major exporter propagates across many importers.
- Vietnam ranks #3, making it an important supplier rather than a passive player. 